In [2]:
import sys
import os
import matplotlib.pyplot as plt
import torch
import gc

### Loading ENV VAR 

In [3]:
from dotenv import load_dotenv
load_dotenv(override=True)  # ensure new values overwrite old ones

# GENERAL PATHS
PROJECT_ROOT = os.getenv("PROJECT_ROOT")

# This makes kernel to see for modules in parent directory
sys.path.append(PROJECT_ROOT)
from src.prepare import TACODataProcessor
from src.train import LitterDetectionTrainer
from config.model_config import model_configs

In [3]:
# DATA PATH CONFIGURATION with respect to train.ipynb file
PROCESSED_DATA_MODIFIED = os.getenv("PROCESSED_DATA_PATH_MODIFIED")
DATA_YAML_MODIFIED = os.getenv("DATA_YAML_PATH_MODIFIED")

# RUN PATHS
RUN_PATH = os.getenv('RUN_PATH')

print(PROJECT_ROOT)
print(os.getcwd())
print(PROCESSED_DATA_MODIFIED)
print(DATA_YAML_MODIFIED)

/home/luc/ai-ml/MLOPS/PROJECT
/home/luc/ai-ml/MLOPS/PROJECT/train
/home/luc/ai-ml/MLOPS/PROJECT/data/taco_dataset_processed_modified
/home/luc/ai-ml/MLOPS/PROJECT/config/data_config.yaml


In [4]:
import os
from collections import Counter

# Path to your label folder
label_dir = "/home/luc/ai-ml/MLOPS/PROJECT/data/taco_dataset_processed_modified/train/labels"

# Get all label files
labels = [os.path.join(label_dir, f) for f in os.listdir(label_dir) if f.endswith(".txt")]

# Count occurrences of each class
c = Counter()
for label_file in labels:
    with open(label_file, "r") as f:
        for line in f:
            cid = int(line.strip().split()[0])
            c[cid] += 1

# Print sorted by class_id
for class_id in sorted(c):
    print(f"Class {class_id}: {c[class_id]} instances")

Class 0: 142 instances
Class 1: 1318 instances
Class 2: 821 instances
Class 3: 376 instances
Class 4: 714 instances
Class 5: 662 instances
Class 6: 2214 instances
Class 7: 583 instances
Class 8: 252 instances
Class 9: 356 instances
Class 10: 751 instances
Class 11: 369 instances
Class 12: 2228 instances
Class 13: 145 instances
Class 14: 225 instances
Class 15: 371 instances
Class 16: 304 instances
Class 17: 1419 instances


## Phase - 2: Model Training

1) First initializing dagshub and mlflow for model tracking


In [5]:
# Checking Configuration of yolo
from ultralytics import settings
print(settings)

JSONDict("/home/luc/.config/Ultralytics/settings.json"):
{
  "settings_version": "0.0.6",
  "datasets_dir": "/home/luc/ai-ml/MLOPS/PROJECT/train/datasets",
  "weights_dir": "weights",
  "runs_dir": "runs",
  "uuid": "080dd5c69bca2800a472c9cb03e3a6d3e0cb3ba856fcf24512dcc3a26c6843a7",
  "sync": true,
  "api_key": "",
  "openai_api_key": "",
  "clearml": true,
  "comet": true,
  "dvc": true,
  "hub": true,
  "mlflow": true,
  "neptune": true,
  "raytune": true,
  "tensorboard": false,
  "wandb": false,
  "vscode_msg": true,
  "openvino_msg": true
}


In [4]:
import gc
gc.collect()
torch.cuda.empty_cache()   

### Training All models in loop

In [ ]:
import gc

for config_name, config in model_configs.items():
    print(config_name)
    config = model_configs[config_name]

    trainer = LitterDetectionTrainer(
        data_yaml=DATA_YAML_MODIFIED,
        model_name=config["model_name"],
        experiment_name="/Shared/Ultralytics"
    )

    model, results = trainer.train(
        epochs=config.get("epochs", 100),
        img_size=config.get("img_size", 640),
        batch_size=config.get("batch_size", 16),
        device=config.get("device", "cuda"),
        optimizer=config.get("optimizer", "SGD"),
        lr=config.get("lr", 0.01),
        fraction=config.get("fraction", 1.0),
        patience=config.get("patience", 50),
        augmentation=config.get("augmentation_config", None),
        tags=config.get("tags", None),
        project=RUN_PATH,
        name=config_name  # Use config_name to differentiate runs
    )

    print(f"✅ Completed training for {config_name}\n")

    del model
    del trainer
    del results
    gc.collect()
    torch.cuda.empty_cache()    

yolov8n_baseline_mod


2025/12/10 09:44:41 INFO mlflow.tracking.fluent: Autologging successfully enabled for sklearn.


🚀 Starting training...
Ultralytics 8.3.235 🚀 Python-3.10.19 torch-2.5.1 CUDA:0 (NVIDIA GeForce RTX 4050 Laptop GPU, 6140MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/luc/ai-ml/MLOPS/PROJECT/config/data_config.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=yolov8n_baseline_mod, nbs=64, nms=False, opset=None, optimize=False, optim

2025/12/10 09:44:56 INFO mlflow.tracking.fluent: Autologging successfully enabled for sklearn.


MLflow: logging run_id(d0d31c9f4a654b15a6ac0f7b104a74ad) to https://dagshub.com/abdullahsyed2005/litter-detection.mlflow
MLflow: disable with 'yolo settings mlflow=False'
Image sizes 640 train, 640 val
Using 8 dataloader workers
Logging results to /home/luc/ai-ml/MLOPS/PROJECT/runs/yolov8n_baseline_mod
Starting training for 100 epochs...

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      1/100      2.47G       1.37      4.261      1.256         30        640: 100% ━━━━━━━━━━━━ 263/263 3.9it/s 1:08<0.2ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 54/54 2.4it/s 22.4s0.3ss
                   all       1704       4830      0.178      0.107     0.0319     0.0202

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      2/100      2.43G      1.399      3.479      1.267         65        640: 100% ━━━━━━━━━━━━ 263/263 4.5it/s 58.9s<0.2s
                 Class     Image

2025/12/10 11:31:25 INFO mlflow.tracking.fluent: Autologging successfully enabled for sklearn.


🚀 Starting training...
Ultralytics 8.3.235 🚀 Python-3.10.19 torch-2.5.1 CUDA:0 (NVIDIA GeForce RTX 4050 Laptop GPU, 6140MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/luc/ai-ml/MLOPS/PROJECT/config/data_config.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=yolov8s_baseline_mod, nbs=64, nms=False, opset=None, optimize=False, optim

2025/12/10 11:31:35 INFO mlflow.tracking.fluent: Autologging successfully enabled for sklearn.


MLflow: logging run_id(ffdaf76d7b5d40f4871882a655577659) to https://dagshub.com/abdullahsyed2005/litter-detection.mlflow
MLflow: disable with 'yolo settings mlflow=False'
Image sizes 640 train, 640 val
Using 8 dataloader workers
Logging results to /home/luc/ai-ml/MLOPS/PROJECT/runs/yolov8s_baseline_mod
Starting training for 100 epochs...

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      1/100         4G      1.324      3.901      1.284         30        640: 100% ━━━━━━━━━━━━ 263/263 3.3it/s 1:19<0.3ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 7% ╸─────────── 4/54 2.1it/s 1.5s<24.4sWARNING ⚠️ NMS time limit 3.600s exceeded
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 54/54 2.4it/s 22.2s0.3ss
                   all       1704       4830      0.198      0.146     0.0829     0.0592

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instan

In [ ]:
import gc

for config_name, config in model_configs.items():
    config = model_configs[config_name]

    trainer = LitterDetectionTrainer(
        data_yaml=DATA_YAML_MODIFIED,
        model_name=config["model_name"],
        experiment_name="/Shared/Ultralytics"
    )

    model, results = trainer.train(
        epochs=config.get("epochs", 100),
        img_size=config.get("img_size", 640),
        batch_size=config.get("batch_size", 16),
        device=config.get("device", "cuda"),
        optimizer=config.get("optimizer", "SGD"),
        lr=config.get("lr", 0.01),
        fraction=config.get("fraction", 1.0),
        patience=config.get("patience", 50),
        augmentation=config.get("augmentation_config", None),
        tags=config.get("tags", None),
        project=RUN_PATH,
        name=config_name  # Use config_name to differentiate runs
    )

    print(f"✅ Completed training for {config_name}\n")

    del model
    del trainer
    del results
    gc.collect()
    torch.cuda.empty_cache()   

2025/12/10 11:59:21 INFO mlflow.tracking.fluent: Autologging successfully enabled for sklearn.


🚀 Starting training...
Ultralytics 8.3.235 🚀 Python-3.10.19 torch-2.5.1 CUDA:0 (NVIDIA GeForce RTX 4050 Laptop GPU, 6140MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/luc/ai-ml/MLOPS/PROJECT/config/data_config.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=yolov8s_baseline_mod, nbs=64, nms=False, opset=None, optimize=False, optim

2025/12/10 11:59:35 INFO mlflow.tracking.fluent: Autologging successfully enabled for sklearn.


MLflow: logging run_id(f4909b331d2d4f4e9eb9d7a237a25a24) to https://dagshub.com/abdullahsyed2005/litter-detection.mlflow
MLflow: disable with 'yolo settings mlflow=False'
Image sizes 640 train, 640 val
Using 8 dataloader workers
Logging results to /home/luc/ai-ml/MLOPS/PROJECT/runs/yolov8s_baseline_mod
Starting training for 100 epochs...

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      1/100         4G      1.324      3.901      1.284         30        640: 100% ━━━━━━━━━━━━ 263/263 3.1it/s 1:24<0.3ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 54/54 2.5it/s 21.6s0.3ss
                   all       1704       4830      0.199      0.147     0.0831     0.0594

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      2/100      3.68G      1.238      2.596      1.193         65        640: 100% ━━━━━━━━━━━━ 263/263 3.4it/s 1:17<0.2ss
                 Class     Image

2025/12/10 14:38:15 INFO mlflow.tracking.fluent: Autologging successfully enabled for sklearn.


🚀 Starting training...
Ultralytics 8.3.235 🚀 Python-3.10.19 torch-2.5.1 CUDA:0 (NVIDIA GeForce RTX 4050 Laptop GPU, 6140MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/luc/ai-ml/MLOPS/PROJECT/config/data_config.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=yolov8s_adam_optimizer_mod, nbs=64, nms=False, opset=None, optimize=False

2025/12/10 14:38:28 INFO mlflow.tracking.fluent: Autologging successfully enabled for sklearn.


MLflow: logging run_id(50a990caf246483e971f187ec8fd1767) to https://dagshub.com/abdullahsyed2005/litter-detection.mlflow
MLflow: disable with 'yolo settings mlflow=False'
Image sizes 640 train, 640 val
Using 8 dataloader workers
Logging results to /home/luc/ai-ml/MLOPS/PROJECT/runs/yolov8s_adam_optimizer_mod
Starting training for 100 epochs...

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      1/100      4.02G      1.432      3.305      1.349         30        640: 100% ━━━━━━━━━━━━ 263/263 3.3it/s 1:20<0.2ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 54/54 3.2it/s 17.1s0.3s
                   all       1704       4830       0.23      0.145     0.0554     0.0352

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      2/100      3.74G      1.449      2.828      1.368         65        640: 100% ━━━━━━━━━━━━ 263/263 3.5it/s 1:16<0.2ss
                 Class     

In [9]:
import mlflow 
from ultralytics import YOLO

model = YOLO("/home/luc/ai-ml/MLOPS/PROJECT/runs/yolov8s_baseline_mod/weights/best.pt")
with mlflow.start_run(run_id="f4909b331d2d4f4e9eb9d7a237a25a24"):
    mlflow.pytorch.log_model(
        pytorch_model=model.model,
        artifact_path="model"
    )
del model

2025/12/10 17:57:51 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


🏃 View run yolo_20251210_115919 at: https://dagshub.com/abdullahsyed2005/litter-detection.mlflow/#/experiments/7/runs/f4909b331d2d4f4e9eb9d7a237a25a24
🧪 View experiment at: https://dagshub.com/abdullahsyed2005/litter-detection.mlflow/#/experiments/7


In [8]:
from mlflow.tracking import MlflowClient

client = MlflowClient()
run_id = "50a990caf246483e971f187ec8fd1767"  # your failed run

# Set the status to FINISHED (success)
client.set_terminated(run_id, status="FINISHED")
print(f"Run {run_id} marked as FINISHED")

🏃 View run yolo_20251210_143813 at: https://dagshub.com/abdullahsyed2005/litter-detection.mlflow/#/experiments/7/runs/50a990caf246483e971f187ec8fd1767
🧪 View experiment at: https://dagshub.com/abdullahsyed2005/litter-detection.mlflow/#/experiments/7
Run 50a990caf246483e971f187ec8fd1767 marked as FINISHED


In [ ]:
# Code to Restore previously deleted experiment in mlflow
from mlflow import MlflowClient, set_experiment, get_experiment_by_name

# Name used previously that got deleted
exp_name = "litter-detection-taco"

client = MlflowClient()
exp = client.get_experiment_by_name(exp_name)

if exp is None:
    # Experiment truly doesn't exist → create a new one
    print(f"No experiment named '{exp_name}' found — creating fresh one.")
    set_experiment(exp_name)
else:
    if exp.lifecycle_stage == 'deleted':
        # Restore the experiment
        print(f"Experiment '{exp_name}' is deleted — restoring it.")
        client.restore_experiment(exp.experiment_id)
    # Now activate it
    set_experiment(exp_name)
    print(f"Using experiment '{exp_name}' (id = {client.get_experiment_by_name(exp_name).experiment_id})")


Experiment 'litter-detection-taco' is deleted — restoring it.
Using experiment 'litter-detection-taco' (id = 0)
